# Notebook 3: Anomaly Detection Experiments
Parameter sweeps for loitering/intrusion thresholds; AUC-ROC on UCF-Crime.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from anomaly.loitering_detector import detect_loitering
from anomaly.intrusion_detector import detect_intrusion
from evaluation.anomaly_metrics import evaluate_anomaly_detection
from utils.types import TrackSnapshot
from utils.config_loader import get_config

config = get_config('../configs/config.yaml')
print('Config loaded')

In [ ]:
# --- 1. Synthetic loitering detection sweep ---
# Create a synthetic track that stays in place for N seconds
def make_loitering_track(dwell_sec, fps=25):
    snaps = []
    for i in range(int(dwell_sec * fps)):
        # Slight jitter around centroid (200, 300)
        cx = 200 + np.random.uniform(-30, 30)
        cy = 300 + np.random.uniform(-30, 30)
        snaps.append(TrackSnapshot(
            frame_index=i, timestamp_sec=i/fps,
            centroid_xy=(cx, cy), bbox_xyxy=[cx-20, cy-40, cx+20, cy+40],
            class_name='person'
        ))
    return snaps

thresholds = [5, 8, 10, 15, 20]  # dwell_time_sec
for thresh in thresholds:
    snaps = make_loitering_track(dwell_sec=12, fps=25)
    events = detect_loitering({1: snaps}, fps=25, dwell_radius_px=80, dwell_time_sec=thresh)
    print(f'  dwell_threshold={thresh}s → {len(events)} events detected')

In [ ]:
# --- 2. Intrusion detection with polygon ---
roi = [[[100, 100], [400, 100], [400, 400], [100, 400]]]  # 300x300 square zone

# Track entering the zone at frame 50
snaps = []
for i in range(100):
    cx = 50 + i * 4  # moves right, enters zone around frame 12
    cy = 250
    snaps.append(TrackSnapshot(
        frame_index=i, timestamp_sec=i/25.0,
        centroid_xy=(cx, cy), bbox_xyxy=[cx-20, cy-40, cx+20, cy+40],
        class_name='person'
    ))

events = detect_intrusion({1: snaps}, roi_polygons=roi, fps=25)
print(f'Intrusion events detected: {len(events)}')
for e in events:
    print(f'  Track {e.track_id}: {e.start_sec:.2f}s – {e.end_sec:.2f}s')

In [ ]:
# --- 3. AUC-ROC evaluation on UCF-Crime (if data available) ---
# Requires: frame-level GT labels + VadCLIP scores from pipeline
# Replace with real data when available

# Simulated: 0=normal, 1=anomaly
np.random.seed(42)
n = 1000
y_true = np.concatenate([np.zeros(700), np.ones(300)]).astype(int)
# Slightly better than random scores
y_scores = np.where(y_true == 1, np.random.beta(5, 2, n), np.random.beta(2, 5, n))

metrics = evaluate_anomaly_detection(y_true.tolist(), y_scores.tolist())
print('Anomaly metrics (synthetic):')
for k, v in metrics.items():
    print(f'  {k}: {v}')